# Network Planning Tower – Interactive Visualizations (Local, No Login)

Loads curated star schema CSVs from `data/curated/`, merges them, calculates
`Net_Position`, and renders **4 interactive Plotly charts entirely inside
Jupyter — no Power BI login required**.

**Run order:** Cell 1 → Cell 2 → Cell 3

In [ ]:
# ── Cell 1 · Imports ──────────────────────────────────────────────────────────
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ── Cell 2 · Load, merge, and calculate Net_Position ─────────────────────────

# Resolve curated/ folder two levels up from this notebook
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
CURATED = os.path.normpath(os.path.join(NOTEBOOK_DIR, "..", "..", "data", "curated"))

def load(name):
    df = pd.read_csv(os.path.join(CURATED, f"{name}.csv"))
    print(f"  {name}: {len(df):,} rows, {len(df.columns)} cols")
    return df

print("Loading curated CSVs ...")
fact_inv  = load("Fact_Inventory")    # DateKey, SKUKey, LocationKey + planning measures
fact_ib   = load("Fact_Inbound")      # DeliveryDateKey, SKUKey, LocationKey + PO measures
fact_tr   = load("Fact_Transfers")    # empty in this snapshot; schema-ready
fact_dem  = load("Fact_Demand")       # ShipDateKey, SKUKey, LocationKey + order measures
dim_prod  = load("Dim_Product")
dim_loc   = load("Dim_Location")

# Normalise date key name across all facts
fact_ib  = fact_ib.rename(columns={"DeliveryDateKey": "DateKey"})
fact_dem = fact_dem.rename(columns={"ShipDateKey": "DateKey"})

# Aggregate Fact_Inbound from PO-line grain → DateKey x SKUKey x LocationKey
ib_agg = (
    fact_ib
    .groupby(["DateKey", "SKUKey", "LocationKey"], as_index=False)
    .agg(
        Inbound_ExpectedQty = ("ExpectedQty", "sum"),
        Inbound_ReceivedQty = ("ReceivedQty", "sum"),
    )
)

# Aggregate Fact_Demand from order-line grain → DateKey x SKUKey x LocationKey
dem_agg = (
    fact_dem
    .groupby(["DateKey", "SKUKey", "LocationKey"], as_index=False)
    .agg(
        Demand_OriginalQty    = ("OriginalQty",    "sum"),
        Demand_OutstandingQty = ("OutstandingQty", "sum"),
    )
)

# Handle Fact_Transfers (empty now; mapped when data is present)
if not fact_tr.empty and "SKUCode" in fact_tr.columns:
    code_to_key = dim_prod.set_index("SKUCode")["SKUKey"].to_dict()
    loc_to_key  = dim_loc.set_index("LocationCode")["LocationKey"].to_dict()
    fact_tr["SKUKey"]      = fact_tr["SKUCode"].map(code_to_key)
    fact_tr["LocationKey"] = fact_tr["OriginLocationCode"].map(loc_to_key)
    fact_tr["DateKey"]     = pd.to_datetime(fact_tr["ShipDate"]).dt.strftime("%Y%m%d").astype(int)
    tr_agg = fact_tr.groupby(["DateKey", "SKUKey", "LocationKey"], as_index=False).agg(
        Transfer_ShipQty=("TransferQty", "sum")
    )
else:
    tr_agg = pd.DataFrame(columns=["DateKey", "SKUKey", "LocationKey", "Transfer_ShipQty"])

# Outer-join all facts on the shared grain
join_keys = ["DateKey", "SKUKey", "LocationKey"]
merged = (
    fact_inv
    .merge(ib_agg,  on=join_keys, how="outer")
    .merge(dem_agg, on=join_keys, how="outer")
    .merge(tr_agg,  on=join_keys, how="outer")
)

# Join dimension descriptors
merged = merged.merge(
    dim_prod[["SKUKey", "SKUCode", "ProductDescription", "Commodity", "ProductGroup"]],
    on="SKUKey", how="left"
)
merged = merged.merge(
    dim_loc[["LocationKey", "LocationCode", "LocationType"]],
    on="LocationKey", how="left"
)

# Fill missing numeric values with 0
num_cols = merged.select_dtypes(include="number").columns
merged[num_cols] = merged[num_cols].fillna(0)

# Calculate Net_Position
# Formula: (InventoryOnHand + Inbound_ReceivedQty + TransfersInReceived)
#        - (GoodSalesShipping + TransferShippingOut + Donations)
# Note: Build_Rework_Qty is not isolated in this ETL run (absorbed into TotalSupply)
merged["Net_Position"] = (
    (merged["InventoryOnHand"] + merged["Inbound_ReceivedQty"] + merged["TransfersInReceived"])
    - (merged["GoodSalesShipping"] + merged["TransferShippingOut"] + merged["Donations"])
)
merged["Shortage_Flag"] = (merged["Net_Position"] < 0).astype(int)

# Convert DateKey integer (YYYYMMDD) to proper date
merged["Date"] = pd.to_datetime(merged["DateKey"].astype(int).astype(str), format="%Y%m%d")

# Final display DataFrame
df_viz = merged[[
    "Date", "SKUCode", "ProductDescription", "Commodity", "ProductGroup",
    "LocationCode", "LocationType",
    "InventoryOnHand", "Inbound_ExpectedQty", "Inbound_ReceivedQty",
    "TransfersInExpected", "TransfersInReceived", "TotalSupply",
    "GoodSalesShipping", "Demand_OriginalQty", "Demand_OutstandingQty",
    "Donations", "TransferShippingOut",
    "Net_Position", "Shortage_Flag", "NetAvailable",
]].copy()

print(f"\nFinal DataFrame: {len(df_viz):,} rows x {len(df_viz.columns)} cols")
print(f"Shortage rows  : {df_viz['Shortage_Flag'].sum():,} of {len(df_viz):,}")
df_viz.head()

In [ ]:
# ── Cell 3 · Four Interactive Plotly Charts ───────────────────────────────────
# No login. No internet. All charts render inline in Jupyter.
# Interact: hover for tooltips | scroll to zoom | drag to pan | click legend to isolate series

RED      = "#D7263D"
GREEN    = "#2DC653"
BLUE     = "#1F77B4"
TEMPLATE = "plotly_white"

# ─────────────────────────────────────────────────────────────────────────────
# CHART 1 – Daily Net Position by Commodity
# Answers: Which commodities are short, and on which days?
# ─────────────────────────────────────────────────────────────────────────────
c1 = (
    df_viz[df_viz["Commodity"].notna()]
    .groupby(["Date", "Commodity"], as_index=False)
    .agg(Net_Position=("Net_Position", "sum"))
)

fig1 = px.bar(
    c1,
    x="Date",
    y="Net_Position",
    color="Commodity",
    barmode="group",
    title="Chart 1 – Daily Net Position by Commodity",
    labels={"Net_Position": "Net Position (units)", "Date": "Date"},
    template=TEMPLATE,
)
fig1.add_hline(
    y=0, line_dash="dash", line_color=RED, line_width=2,
    annotation_text="Shortage threshold",
    annotation_position="top left",
)
fig1.update_layout(hovermode="x unified", legend_title="Commodity")
fig1.show()


# ─────────────────────────────────────────────────────────────────────────────
# CHART 2 – Supply vs Demand over Time  (with Net Position panel)
# Answers: Where does demand exceed supply across the planning horizon?
# ─────────────────────────────────────────────────────────────────────────────
c2 = (
    df_viz
    .groupby("Date", as_index=False)
    .agg(
        Total_Supply = ("TotalSupply",        "sum"),
        Total_Demand = ("Demand_OriginalQty", "sum"),
        Net_Pos      = ("Net_Position",        "sum"),
    )
)

fig2 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Supply vs Demand (daily totals)", "Net Position"),
    row_heights=[0.65, 0.35],
    vertical_spacing=0.08,
)
fig2.add_trace(
    go.Bar(x=c2["Date"], y=c2["Total_Supply"], name="Total Supply",
           marker_color=BLUE, opacity=0.85),
    row=1, col=1,
)
fig2.add_trace(
    go.Scatter(x=c2["Date"], y=c2["Total_Demand"], name="Total Demand",
               mode="lines+markers", line=dict(color=RED, width=2)),
    row=1, col=1,
)
fig2.add_trace(
    go.Bar(
        x=c2["Date"],
        y=c2["Net_Pos"],
        name="Net Position",
        marker_color=[RED if v < 0 else GREEN for v in c2["Net_Pos"]],
    ),
    row=2, col=1,
)
fig2.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)
fig2.update_layout(
    title_text="Chart 2 – Supply vs Demand over Time",
    template=TEMPLATE,
    hovermode="x unified",
)
fig2.show()


# ─────────────────────────────────────────────────────────────────────────────
# CHART 3 – Top 15 SKUs Deepest in Shortage
# Answers: Which SKUs need urgent attention? (cumulative across all dates)
# ─────────────────────────────────────────────────────────────────────────────
c3 = (
    df_viz[df_viz["SKUCode"].notna()]
    .groupby(["SKUCode", "ProductDescription"], as_index=False)
    .agg(Cumulative_Net_Position=("Net_Position", "sum"))
    .sort_values("Cumulative_Net_Position")
    .head(15)
)
c3["Label"] = c3["SKUCode"] + "  " + c3["ProductDescription"].str[:45]

fig3 = px.bar(
    c3,
    x="Cumulative_Net_Position",
    y="Label",
    orientation="h",
    color="Cumulative_Net_Position",
    color_continuous_scale=[RED, "#FFCDD2"],
    title="Chart 3 – Top 15 SKUs Deepest in Shortage (cumulative across all dates)",
    labels={"Cumulative_Net_Position": "Cumulative Net Position (units)", "Label": ""},
    template=TEMPLATE,
)
fig3.add_vline(x=0, line_dash="dash", line_color="grey")
fig3.update_layout(
    coloraxis_showscale=False,
    yaxis={"autorange": "reversed"},
    height=500,
)
fig3.show()


# ─────────────────────────────────────────────────────────────────────────────
# CHART 4 – Inbound Fill Rate by Location
# Answers: Which receiving locations have supplier delivery gaps?
# ─────────────────────────────────────────────────────────────────────────────
c4 = (
    ib_agg
    .merge(dim_loc[["LocationKey", "LocationCode", "LocationType"]], on="LocationKey", how="left")
    .groupby(["LocationCode", "LocationType"], as_index=False)
    .agg(
        Total_Expected = ("Inbound_ExpectedQty", "sum"),
        Total_Received = ("Inbound_ReceivedQty", "sum"),
    )
)
c4["Fill_Rate_Pct"] = (
    c4["Total_Received"] / c4["Total_Expected"].replace(0, float("nan"))
) * 100
c4 = c4.dropna(subset=["Fill_Rate_Pct"]).sort_values("Fill_Rate_Pct")

fig4 = px.bar(
    c4,
    x="Fill_Rate_Pct",
    y="LocationCode",
    orientation="h",
    color="Fill_Rate_Pct",
    color_continuous_scale=[RED, "#FFF176", GREEN],
    range_color=[80, 105],
    title="Chart 4 – Inbound Fill Rate by Location (Received / Expected)",
    labels={"Fill_Rate_Pct": "Fill Rate (%)", "LocationCode": "Location"},
    hover_data={"Total_Expected": True, "Total_Received": True},
    template=TEMPLATE,
)
fig4.add_vline(
    x=100, line_dash="dash", line_color="grey",
    annotation_text="100%", annotation_position="top",
)
fig4.update_layout(
    coloraxis_showscale=False,
    yaxis={"autorange": "reversed"},
    height=max(350, len(c4) * 28),
)
fig4.show()